# Laboratorio de métodos de búsqueda

|                |                  |
:----------------|------------------|
| **Nombre**     | Omar Mendoza     |
| **Fecha**      | 29/10/2026       |
| **Expediente** | 757806           |

Modelos como regresión logística, máquinas de vectores de soporte, y preceptrones multicapa tienen una serie de argumentos que necesitan valores dados antes de poder usarse para ajustarse a un dataset. Estos argumentos se llaman hiperparámetros.

¿Cuál es la diferencia entre un parámetro y un hiperparámetro?

Los parametros los aprende el modelo solo al entrenar, como los betas de la regresion o los pesos de la red. Los hiperparametros son los que se ponen antes de entrenar, como el alpha, el kernel o cuantas neuronas lleva cada capa

Este laboratorio se centra en métodos de búsqueda para encontrar los mejores hiperparámetros para un modelo dado. Imagina que estás jugando batalla naval:

<center><img src="https://cdn11.bigcommerce.com/s-nehqfptb/images/stencil/670x993/products/40096/100569/battleship-game__73281.1768238533.jpg?c=2"></center>

Una forma de jugar podría ser ir de cuadro en cuadro hasta probar todo el tablero. El problema aquí es que se juega a contrarreloj, debido a que tu oponente puede hundir tus barcos antes de que pruebes todo el tablero.

Si no se tiene la preocupación/limitante de tiempo, probar todas las combinaciones es una opción:

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

Entra a la documentación de `GridSearchCV` de `sklearn` y describe cómo funciona este objeto (entradas, salidas, parámetros).

Prueba todas las combinaciones de hiperparametros que le das, a cada una le hace cross validation y se queda con la que saco mejor promedio. Despues reentrena esa con todos los datos

Entradas:

- estimator: el modelo o el pipeline
- param_grid: diccionario con los valores a probar de cada hiperparametro, si es pipeline el nombre va como paso__parametro
- scoring: el metrico para calificar, por ejemplo "roc_auc"
- cv: cuantos folds o un KFold
- n_jobs: cuantos nucleos usar
- refit: si reentrena la mejor al final, default True

Salidas:

- best_params_: la mejor combinacion
- best_score_: su calificacion promedio
- best_estimator_: el modelo ya entrenado con la mejor combinacion
- cv_results_: los resultados de todas las combinaciones

Otra forma de "buscar" podría ser tirar al azar y ver qué pasa. Esto sería otra forma de utilizar la fuerza bruta para encontrar información. 

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html

Entra a la documentación de `RandomizedSearchCV` de `sklearn` y describe cómo funciona este objeto (entradas, salidas, parámetros). 

Funciona igual que GridSearchCV pero no prueba todo, escoge combinaciones al azar

Entradas: las mismas que GridSearchCV pero en vez de param_grid lleva param_distributions, que puede tener listas o distribuciones de scipy. Ademas lleva n_iter que es cuantas combinaciones al azar va a probar, default 10, y random_state para que salgan las mismas

Salidas: las mismas, best_params_, best_score_, best_estimator_ y cv_results_

Sirve cuando hay demasiadas combinaciones y no da tiempo de probarlas todas

**Nota:** RandomizedSearchCV prueba puramente al azar. Si encontrara en uno de sus tiros que se atinó a un barco, no cambia el algoritmo de búsqueda para seguir buscando en esa zona. Para un proceso de búsqueda más refinado necesitaríamos un algoritmo que cambie su forma de decidir a partir de información anterior.

Revisa la documentación de Optuna (https://optuna.org/). Instala la librería y escribe una guía de cómo se debe utlizar para un dataset dado.

Optuna busca hiperparametros aprendiendo de los intentos anteriores. Si una zona dio buen resultado prueba mas cerca de ahi, como cuando le atinas a un barco y tiras alrededor. Por default usa un metodo que se llama TPE

Guia para usarlo con un dataset:

1. Instalar con pip install optuna
2. Separar los datos en train y test
3. Hacer una funcion objetivo que recibe un trial. Adentro se escogen los hiperparametros con trial.suggest_int, trial.suggest_float o trial.suggest_categorical, se entrena el modelo con cross validation y se regresa el metrico
4. Crear el estudio con optuna.create_study(direction="maximize") si el metrico es mejor entre mas alto
5. Correr study.optimize(objetivo, n_trials=50) con cuantos intentos quieres
6. Ver los mejores con study.best_params y study.best_value
7. Entrenar el modelo final con esos hiperparametros y probarlo en test

Ejemplo:

```python
import optuna

def objetivo(trial):
    alpha = trial.suggest_float("alpha", 0.0001, 1)
    modelo = MLPClassifier(alpha=alpha)
    # entrenar con cross validation y regresar el AUC promedio
    return auc

study = optuna.create_study(direction="maximize")
study.optimize(objetivo, n_trials=50)
print(study.best_params)
```

### Ejercicio

Utiliza el dataset `College.csv` para intentar predecir si una universidad es privada a partir del resto de los factores.

Utiliza un perceptrón multicapa con hiperparámetros:
- Número de capas intermedias --> {2, 3, 4}
- Número de neuronas en capa 1 --> [5, 10]
- Número de neuronas en capa 2 --> [10, 20]
- Número de neuronas en capa 3 (si existe) --> [5, 20]
- Número de neuronas en capa 4 (si existe) --> [5, 10]

Utiliza k-folds cross-validation con $k=10$; optimiza el ROC-AUC de test.

Compara los 3 diferentes métodos de búsqueda descritos en este laboratorio.

In [1]:
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import train_test_split, KFold, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [2]:
datos = pd.read_csv("College.csv", index_col=0)

print(datos.shape)
print("Nulos:", datos.isna().sum().sum())
print("Duplicados:", datos.duplicated().sum())
print(datos["Private"].value_counts())
datos.head()

(777, 18)
Nulos: 0
Duplicados: 0
Private
Yes    565
No     212
Name: count, dtype: int64


,Private,Apps,Accept,Enroll,Top10perc,Top25perc,F.Undergrad,P.Undergrad,Outstate,Room.Board,Books,Personal,PhD,Terminal,S.F.Ratio,perc.alumni,Expend,Grad.Rate
Abilene Christian University,Yes,1660,1232,721,23,52,2885,537,7440,3300,450,2200,70,78,18.1,12,7041,60
Adelphi University,Yes,2186,1924,512,16,29,2683,1227,12280,6450,750,1500,29,30,12.2,16,10527,56
Adrian College,Yes,1428,1097,336,22,50,1036,99,11250,3750,400,1165,53,66,12.9,30,8735,54
Agnes Scott College,Yes,417,349,137,60,89,510,63,12960,5450,450,875,92,97,7.7,37,19016,59
Alaska Pacific University,Yes,193,146,55,16,44,249,869,7560,4120,800,1500,76,72,11.9,2,10922,15


Son 777 universidades con 17 columnas numericas y Private que es lo que se quiere predecir. No hay nulos ni duplicados. Hay 565 privadas y 212 publicas, no estan parejas y por eso conviene el AUC

In [3]:
X = datos.drop(columns="Private")
y = (datos["Private"] == "Yes").astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

pipeline = Pipeline(steps=[("esc", StandardScaler()), ("mlp", MLPClassifier(random_state=42))])

kf = KFold(n_splits=10, shuffle=True, random_state=42)

print(X_train.shape, X_test.shape)

(543, 17) (234, 17)


Quedan 543 universidades para entrenar y 234 para test. Todo se escala en el pipeline y los tres metodos usan el mismo KFold de 10 para que la comparacion sea justa

In [4]:
capas = []

for c1 in [5, 10]:
    for c2 in [10, 20]:
        capas.append((c1, c2))
        for c3 in [5, 20]:
            capas.append((c1, c2, c3))
            for c4 in [5, 10]:
                capas.append((c1, c2, c3, c4))

print(len(capas))
print(capas)

28
[(5, 10), (5, 10, 5), (5, 10, 5, 5), (5, 10, 5, 10), (5, 10, 20), (5, 10, 20, 5), (5, 10, 20, 10), (5, 20), (5, 20, 5), (5, 20, 5, 5), (5, 20, 5, 10), (5, 20, 20), (5, 20, 20, 5), (5, 20, 20, 10), (10, 10), (10, 10, 5), (10, 10, 5, 5), (10, 10, 5, 10), (10, 10, 20), (10, 10, 20, 5), (10, 10, 20, 10), (10, 20), (10, 20, 5), (10, 20, 5, 5), (10, 20, 5, 10), (10, 20, 20), (10, 20, 20, 5), (10, 20, 20, 10)]


Salen 28 redes, 4 de 2 capas, 8 de 3 capas y 16 de 4 capas. Tome los valores de cada capa como las dos opciones que dice el ejercicio, 5 o 10 en la primera, 10 o 20 en la segunda y asi

In [5]:
inicio = time.time()

grid = GridSearchCV(pipeline, {"mlp__hidden_layer_sizes": capas}, scoring="roc_auc", cv=kf)
grid.fit(X_train, y_train)

tiempo_grid = time.time() - inicio

print("Mejor:", grid.best_params_)
print("AUC CV:", grid.best_score_)
print("Tiempo:", tiempo_grid)

Mejor: {'mlp__hidden_layer_sizes': (10, 20, 5)}
AUC CV: 0.9793017380784267
Tiempo: 175.35794687271118


GridSearchCV probo las 28 redes con los 10 folds, 280 entrenamientos, y tardo 175 segundos. La mejor fue la de 10, 20 y 5 neuronas con 0.979 de AUC

In [6]:
inicio = time.time()

random = RandomizedSearchCV(pipeline, {"mlp__hidden_layer_sizes": capas}, n_iter=10, scoring="roc_auc", cv=kf, random_state=42)
random.fit(X_train, y_train)

tiempo_random = time.time() - inicio

print("Mejor:", random.best_params_)
print("AUC CV:", random.best_score_)
print("Tiempo:", tiempo_random)

Mejor: {'mlp__hidden_layer_sizes': (10, 20, 5)}
AUC CV: 0.9793017380784267
Tiempo: 60.0360746383667


RandomizedSearchCV solo probo 10 redes al azar y tardo 60 segundos, casi la tercera parte. Le atino a la misma red que el grid, la de 10, 20 y 5 con 0.979, pero fue suerte, con otra semilla pudo no caer ahi

In [7]:
def objetivo(trial):
    n_capas = trial.suggest_int("n_capas", 2, 4)
    c1 = trial.suggest_categorical("c1", [5, 10])
    c2 = trial.suggest_categorical("c2", [10, 20])
    c3 = trial.suggest_categorical("c3", [5, 20])
    c4 = trial.suggest_categorical("c4", [5, 10])

    red = tuple([c1, c2, c3, c4][:n_capas])
    pipeline.set_params(mlp__hidden_layer_sizes=red)

    aucs = []
    for train_idx, test_idx in kf.split(X_train):
        pipeline.fit(X_train.iloc[train_idx], y_train.iloc[train_idx])
        proba = pipeline.predict_proba(X_train.iloc[test_idx])[:, 1]
        aucs.append(roc_auc_score(y_train.iloc[test_idx], proba))

    return np.mean(aucs)

Esta es la funcion que Optuna va a maximizar. Escoge cuantas capas y las neuronas de cada una, arma la red y regresa el AUC promedio de los 10 folds. Si la red es de 2 capas solo usa c1 y c2

In [8]:
inicio = time.time()

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objetivo, n_trials=10)

tiempo_optuna = time.time() - inicio

p = study.best_params
red_optuna = tuple([p["c1"], p["c2"], p["c3"], p["c4"]][:p["n_capas"]])

print("Mejor:", red_optuna)
print("AUC CV:", study.best_value)
print("Tiempo:", tiempo_optuna)

Mejor: (10, 10)
AUC CV: 0.9745433731940067
Tiempo: 56.83533716201782


Optuna hizo 10 intentos y tardo 57 segundos. Su mejor fue la de 10 y 10 neuronas con 0.975, un poquito abajo del grid. Con tan pocas opciones no le da tiempo de aprovechar lo que aprende de los intentos anteriores

In [9]:
# AUC de test de la mejor red de cada metodo

auc_grid = roc_auc_score(y_test, grid.best_estimator_.predict_proba(X_test)[:, 1])
auc_random = roc_auc_score(y_test, random.best_estimator_.predict_proba(X_test)[:, 1])

pipeline.set_params(mlp__hidden_layer_sizes=red_optuna)
pipeline.fit(X_train, y_train)
auc_optuna = roc_auc_score(y_test, pipeline.predict_proba(X_test)[:, 1])

pd.DataFrame({
    "metodo": ["GridSearchCV", "RandomizedSearchCV", "Optuna"],
    "redes probadas": [len(capas), 10, 10],
    "mejor red": [grid.best_params_["mlp__hidden_layer_sizes"], random.best_params_["mlp__hidden_layer_sizes"], red_optuna],
    "AUC CV": [grid.best_score_, random.best_score_, study.best_value],
    "AUC test": [auc_grid, auc_random, auc_optuna],
    "segundos": [tiempo_grid, tiempo_random, tiempo_optuna],
}).round(4)

,metodo,redes probadas,mejor red,AUC CV,AUC test,segundos
0,GridSearchCV,28,"(10, 20, 5)",0.9793,0.9636,175.3579
1,RandomizedSearchCV,10,"(10, 20, 5)",0.9793,0.9636,60.0361
2,Optuna,10,"(10, 10)",0.9745,0.9730,56.8353


Los tres quedan muy parecidos, todos arriba de 0.97 de AUC en CV y arriba de 0.96 en test, el modelo separa muy bien privadas de publicas

GridSearchCV es el unico que asegura encontrar la mejor pero es el mas lento, tardo casi 3 veces mas. Random y Optuna tardaron como un minuto con solo 10 intentos. Random tuvo suerte y encontro la misma que el grid, y Optuna saco un poco menos en CV pero el mejor AUC de test, 0.973 contra 0.964, aunque esa diferencia es chica

Aqui con 28 redes el grid todavia se puede correr, pero si fueran todos los numeros de cada rango serian 7458 redes y tardaria horas. Ahi es donde conviene Random u Optuna, y Optuna es mejor porque aprende de lo que ya probo